# TCGA-BRCA OS Endpoint V1 Review

This notebook reviews the saved TCGA-BRCA OS endpoint v1 outputs from disk only. It does not parse raw files, redefine the rule, add treatment detail, or perform modeling.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def detect_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Unable to locate the repository root from the notebook path.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = detect_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'endpoint-prep'
    / 'tcga_brca_os_endpoint_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest OS endpoint v1 pointer not found: {latest_pointer_path}. Run the OS endpoint v1 script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
os_path = repo_root / latest_pointer['os_endpoint_v1_tsv']
conflict_audit_path = repo_root / latest_pointer['os_endpoint_v1_conflict_audit_tsv']
rule_spec_path = repo_root / latest_pointer['os_endpoint_v1_rule_spec_tsv']
summary_path = repo_root / latest_pointer['os_endpoint_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']

for required_path in [os_path, conflict_audit_path, rule_spec_path, summary_path, run_log_path]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required OS endpoint v1 artifact not found: {required_path}')

os_df = read_tsv(os_path)
conflict_audit_df = read_tsv(conflict_audit_path)
rule_spec_df = read_tsv(rule_spec_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

results_root = (
    repo_root
    / '09-trials'
    / '01-tcga-only-source-audited'
    / '05-results'
)
results_root.mkdir(parents=True, exist_ok=True)


In [2]:
os_review_path = results_root / '102_os_endpoint_v1.tsv'
conflict_review_path = results_root / '103_os_endpoint_v1_conflict_audit.tsv'
rule_spec_review_path = results_root / '104_os_endpoint_v1_rule_spec.tsv'
manual_review_path = results_root / '105_os_endpoint_v1_manual_review_cases.tsv'
summary_review_path = results_root / '106_os_endpoint_v1_summary.tsv'

manual_review_df = os_df.loc[
    os_df['os_requires_manual_review'] == 'yes'
].copy()
manual_review_sort_cols = [
    column
    for column in [
        'os_endpoint_inclusion_status',
        'os_conflict_vital_status',
        'os_conflict_dead_without_death_time',
        'provisional_patient_row_id',
    ]
    if column in manual_review_df.columns
]
if manual_review_sort_cols:
    manual_review_df = manual_review_df.sort_values(manual_review_sort_cols, kind='stable').reset_index(drop=True)

event_censor_df = pd.DataFrame(
    [
        {
            'event_bucket': 'os_event_1',
            'patient_count': int((os_df['os_event'] == '1').sum()),
        },
        {
            'event_bucket': 'os_event_0',
            'patient_count': int((os_df['os_event'] == '0').sum()),
        },
        {
            'event_bucket': 'os_event_missing',
            'patient_count': int((os_df['os_event'] == '').sum()),
        },
        {
            'event_bucket': 'os_time_nonmissing',
            'patient_count': int((os_df['os_time_days'] != '').sum()),
        },
        {
            'event_bucket': 'os_time_missing',
            'patient_count': int((os_df['os_time_days'] == '').sum()),
        },
    ]
)

inclusion_status_df = (
    os_df.groupby('os_endpoint_inclusion_status', as_index=False)
    .size()
    .rename(columns={'size': 'patient_count'})
    .sort_values(['patient_count', 'os_endpoint_inclusion_status'], ascending=[False, True])
    .reset_index(drop=True)
)

time_source_distribution_df = (
    os_df.groupby('os_time_source', as_index=False)
    .size()
    .rename(columns={'size': 'patient_count'})
    .sort_values(['patient_count', 'os_time_source'], ascending=[False, True])
    .reset_index(drop=True)
)

conflict_type_counts_df = (
    conflict_audit_df.groupby(['conflict_type', 'manual_review_priority'], as_index=False)
    .size()
    .rename(columns={'size': 'row_count'})
    .sort_values(['manual_review_priority', 'row_count', 'conflict_type'], ascending=[True, False, True])
    .reset_index(drop=True)
)

required_join_columns = [
    'bcr_patient_barcode',
    'bcr_patient_uuid',
    'provisional_patient_row_id',
    'baseline_analysis_v1_row_id',
    'feature_set_v1_row_index',
]
join_readiness_df = pd.DataFrame(
    [
        {
            'os_endpoint_v1_run_id': latest_pointer['os_endpoint_v1_run_id'],
            'row_count': int(os_df.shape[0]),
            'rows_with_complete_join_keys': int((os_df[required_join_columns] != '').all(axis=1).sum()),
            'included_rule_based_clean': int((os_df['os_endpoint_inclusion_status'] == 'included_rule_based_clean').sum()),
            'included_rule_based_conflict_flagged': int((os_df['os_endpoint_inclusion_status'] == 'included_rule_based_conflict_flagged').sum()),
            'excluded_rows': int(os_df['os_endpoint_inclusion_status'].str.startswith('excluded_').sum()),
            'manual_review_rows': int((os_df['os_requires_manual_review'] == 'yes').sum()),
        }
    ]
)

os_df.to_csv(os_review_path, sep='\t', index=False)
conflict_audit_df.to_csv(conflict_review_path, sep='\t', index=False)
rule_spec_df.to_csv(rule_spec_review_path, sep='\t', index=False)
manual_review_df.to_csv(manual_review_path, sep='\t', index=False)
summary_df.to_csv(summary_review_path, sep='\t', index=False)


In [3]:
print(f"OS endpoint v1 run ID: {latest_pointer['os_endpoint_v1_run_id']}")
print(f"Endpoint-target prep v1 run ID: {latest_pointer['endpoint_target_prep_v1_run_id']}")
print(f"Run log: {run_log_path}")
print(f"Saved: {os_review_path}")
print(f"Saved: {conflict_review_path}")
print(f"Saved: {rule_spec_review_path}")
print(f"Saved: {manual_review_path}")
print(f"Saved: {summary_review_path}")

display(pd.DataFrame([latest_pointer]))
display(pd.DataFrame([run_log.get('validation', {})]))
display(event_censor_df)
display(inclusion_status_df)
display(time_source_distribution_df)
display(conflict_type_counts_df)
display(join_readiness_df)
display(manual_review_df.head(20))
display(rule_spec_df)
display(summary_df)


OS endpoint v1 run ID: 20260414T162349Z
Endpoint-target prep v1 run ID: 20260414T025234Z
Run log: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\endpoint-prep\os_endpoint_v1_runs\20260414T162349Z\run_log.json
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\102_os_endpoint_v1.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\103_os_endpoint_v1_conflict_audit.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\104_os_endpoint_v1_rule_spec.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\105_os_endpoint_v1_manual_review_cases.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\106_os_endpoint_v1_summary.tsv


,updated_at_utc,os_endpoint_v1_run_id,endpoint_target_prep_v1_run_id,source_run_id,cohort_v1_build_id,baseline_model_input_v1_run_id,baseline_analysis_v1_run_id,processed_run_directory,audit_run_directory,os_endpoint_v1_tsv,os_endpoint_v1_rule_spec_tsv,os_endpoint_v1_conflict_audit_tsv,os_endpoint_v1_summary_tsv,run_log_json,endpoint_target_prep_v1_latest_json,minimal_cohort_v1_latest_json,baseline_model_input_v1_latest_json
0,2026-04-14T16:23:50Z,20260414T162349Z,20260414T025234Z,20260412T000556Z,20260413T202134Z,20260414T013108Z,20260413T213900Z,01-data/processed/tcga-brca/endpoint-prep/os_e...,01-data/audit/tcga-brca/endpoint-prep/os_endpo...,01-data/processed/tcga-brca/endpoint-prep/os_e...,01-data/audit/tcga-brca/endpoint-prep/os_endpo...,01-data/audit/tcga-brca/endpoint-prep/os_endpo...,01-data/audit/tcga-brca/endpoint-prep/os_endpo...,01-data/audit/tcga-brca/endpoint-prep/os_endpo...,01-data/audit/tcga-brca/endpoint-prep/tcga_brc...,01-data/audit/tcga-brca/cohort/tcga_brca_minim...,01-data/audit/tcga-brca/model-input/tcga_brca_...


,passed,required_upstream_pointers_found,endpoint_target_prep_latest_pointer_found,endpoint_target_prep_run_log_completed,endpoint_target_prep_validation_passed,minimal_cohort_v1_latest_pointer_found,minimal_cohort_v1_run_log_completed,minimal_cohort_v1_validation_passed,baseline_model_input_v1_latest_pointer_found,baseline_model_input_v1_run_log_completed,...,rule_spec_row_count_positive,conflict_audit_row_count_positive,summary_row_count_positive,summary_counts_reconcile_to_os_table,os_inclusion_status_values_valid,followup_versions_restricted_to_supported_set,summary_readiness_matches_expected,summary_treatment_unchanged,no_prior_run_overwrite,latest_pointer_written_after_success_only
0,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True


,event_bucket,patient_count
0,os_event_1,152
1,os_event_0,945
2,os_event_missing,0
3,os_time_nonmissing,1097
4,os_time_missing,0


,os_endpoint_inclusion_status,patient_count
0,included_rule_based_clean,1049
1,included_rule_based_conflict_flagged,48


...

,conflict_type,manual_review_priority,row_count
0,vital_status_disagreement,high,48
1,dead_without_death_time,high,1
2,multiple_source_reconciliation,low,425


,os_endpoint_v1_run_id,row_count,rows_with_complete_join_keys,included_rule_based_clean,included_rule_based_conflict_flagged,excluded_rows,manual_review_rows
0,20260414T162349Z,1097,1097,1049,48,0,48


,os_endpoint_v1_run_id,cohort_v1_build_id,baseline_model_input_v1_run_id,bcr_patient_barcode,bcr_patient_uuid,provisional_patient_row_id,baseline_analysis_v1_row_id,feature_set_v1_row_index,os_event,os_time_days,...,os_header_days_to_death,os_followup_max_days_to_death,os_header_days_to_last_followup,os_followup_max_days_to_last_followup,os_followup_max_days_to_last_known_alive,os_followup_versions_present_json,os_followup_versions_contributing_time_json,os_multiple_followup_versions_contributed,os_followup_max_last_contact_exceeds_header,os_rule_status_flags_json
0,20260414T162349Z,20260413T202134Z,20260414T013108Z,TCGA-HN-A2OB,61204CDF-3B1C-44A1-BDA6-0FEF1AC4A333,1018,1018,1018,1,1900,...,,1900,1849,1883,,"[""4.0""]","[""4.0""]",no,yes,"[""os_conflict_vital_status"", ""os_requires_manu..."
1,20260414T162349Z,20260413T202134Z,20260414T013108Z,TCGA-LL-A73Z,3E637872-F5E5-49D3-BB0D-9C16B8713382,1039,1039,1039,1,227,...,,227,137,,,"[""4.0""]","[""4.0""]",no,no,"[""os_conflict_vital_status"", ""os_requires_manu..."
2,20260414T162349Z,20260413T202134Z,20260414T013108Z,TCGA-OL-A5D6,C3A981C7-F148-4252-BD50-AF8A49EC0DF8,1048,1048,1048,1,1104,...,,1104,385,,,"[""4.0""]","[""4.0""]",no,no,"[""os_conflict_vital_status"", ""os_requires_manu..."
3,20260414T162349Z,20260413T202134Z,20260414T013108Z,TCGA-OL-A66K,2AB1B35B-9E56-475C-AC9B-15ED0A422683,1062,1062,1062,1,1275,...,,1275,1021,,,"[""4.0""]","[""4.0""]",no,no,"[""os_conflict_vital_status"", ""os_requires_manu..."
4,20260414T162349Z,20260413T202134Z,20260414T013108Z,TCGA-A2-A3XX,53886143-C1C6-40E9-88E6-E4E5E0271FC8,114,114,114,1,1439,...,,1439,1168,,,"[""4.0""]","[""4.0""]",no,no,"[""os_conflict_vital_status"", ""os_requires_manu..."
5,20260414T162349Z,20260413T202134Z,20260414T013108Z,TCGA-A2-A3XY,DEBA32E4-0E68-4711-941B-3B63BD965AFB,115,115,115,1,1093,...,,1093,786,1064,,"[""4.0""]","[""4.0""]",no,yes,"[""os_conflict_vital_status"", ""os_requires_manu..."
6,20260414T162349Z,20260413T202134Z,20260414T013108Z,TCGA-A7-A13E,8C7E74E0-71EF-49B8-9217-94B8EF740EF9,135,135,135,1,614,...,,614,287,326,,"[""1.5"", ""2.1""]","[""2.1""]",yes,yes,"[""os_conflict_vital_status"", ""os_requires_manu..."
7,20260414T162349Z,20260413T202134Z,20260414T013108Z,TCGA-A8-A08T,32BC7CC4-D185-4594-BBEE-4B96410BE512,214,214,214,1,3409,...,,3409,2830,,,"[""4.0""]","[""4.0""]",no,no,"[""os_conflict_vital_status"", ""os_requires_manu..."
8,20260414T162349Z,20260413T202134Z,20260414T013108Z,TCGA-AC-A23H,7DCF550C-90CE-4F63-AECD-0E46897E2A3E,254,254,254,1,0,...,,0,0,,,"[""4.0""]","[""4.0""]",no,no,"[""os_conflict_vital_status"", ""os_requires_manu..."
9,20260414T162349Z,20260413T202134Z,20260414T013108Z,TCGA-AC-A2FE,F6EEBD4B-B63A-4A9C-92D3-0D954A8A6655,259,259,259,1,2636,...,,2636,791,2240,,"[""4.0""]","[""4.0""]",no,yes,"[""os_conflict_vital_status"", ""os_requires_manu..."


,os_endpoint_v1_run_id,rule_name,rule_order,rule_description,fields_used,rationale,notes
0,20260414T162349Z,anchor_universe,1,Freeze one OS row per patient in the current m...,"[""endpoint_target_prep_v1.tsv"", ""minimal_cohor...",The OS target must stay join-compatible with t...,"No treatment fields, recurrence fields, modeli..."
1,20260414T162349Z,event_dead_if_any_dead_signal,2,Set os_event=1 if any patient-header or follow...,"[""patient_header_vital_status"", ""followup_v1_5...",The v1 OS freeze uses a conservative death rul...,Death-time evidence itself is treated as death...
2,20260414T162349Z,event_alive_only_if_consistent,3,Set os_event=0 only when all non-missing vital...,"[""patient_header_vital_status"", ""followup_v1_5...",Alive censoring should only be assigned when t...,Patients with no clear status remain flagged i...
3,20260414T162349Z,event_blank_if_no_clear_status,4,Leave os_event blank if neither the death rule...,"[""os_event"", ""os_conflict_time_without_clear_s...",The workflow must keep unresolved cases explic...,These rows remain in the patient-level table b...
4,20260414T162349Z,time_use_max_days_to_death,5,"If any days_to_death value exists, set os_time...","[""patient_header_days_to_death"", ""followup_v1_...",The freeze rule keeps all XML layers in play a...,Tied maxima keep multiple source labels in os_...
5,20260414T162349Z,time_else_use_max_last_contact_like,6,"If no days_to_death exists, set os_time_days t...","[""patient_header_days_to_last_followup"", ""pati...",The v1 OS freeze uses a conservative censoring...,days_to_last_followup remains primary; days_to...
6,20260414T162349Z,conflict_vital_status,7,Flag os_conflict_vital_status=yes when Alive a...,"[""os_header_vital_status"", ""os_followup_vital_...",The workflow must keep cross-source status dis...,Conflict rows retain provisional values under ...
7,20260414T162349Z,conflict_dead_without_death_time,8,Flag os_conflict_dead_without_death_time=yes w...,"[""os_event"", ""os_time_days"", ""os_time_source"",...",This is a weaker event/time pairing and should...,The current saved data is expected to have one...
8,20260414T162349Z,inclusion_status_mapping,9,Assign inclusion status from the resolved even...,"[""os_time_missing"", ""os_conflict_time_without_...","Downstream users need an explicit, auditable s...","Allowed values are included_rule_based_clean, ..."


,os_endpoint_v1_run_id,summary_section,summary_metric,summary_value,notes
0,20260414T162349Z,input_runs,endpoint_target_prep_v1_run_id,20260414T025234Z,
1,20260414T162349Z,input_runs,cohort_v1_build_id,20260413T202134Z,
2,20260414T162349Z,input_runs,baseline_model_input_v1_run_id,20260414T013108Z,
3,20260414T162349Z,input_runs,source_run_id,20260412T000556Z,
4,20260414T162349Z,row_counts,os_endpoint_v1_row_count,1097,
5,20260414T162349Z,row_counts,os_endpoint_v1_conflict_audit_row_count,474,
6,20260414T162349Z,row_counts,os_endpoint_v1_rule_spec_row_count,9,
7,20260414T162349Z,row_counts,clinical_xml_patient_endpoint_fields_row_count,1097,
8,20260414T162349Z,row_counts,clinical_xml_followup_fields_long_row_count,16332,
9,20260414T162349Z,event_counts,patients_with_os_event_1,152,
